# 🕵️ Dia 27 — Forense de metadados, parte 1: imagens e geolocalização

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 27: Metadata forensics, part 1: images and geolocation* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-27.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Quarta, 7 de outubro. Circula nos grupos de WhatsApp da cidade uma foto da fachada da agência Centro da Serra Azul com uma faixa vermelha: *"ATENDIMENTO SUSPENSO — REGULARIZE SUA CHAVE PIX"*. Um cooperado mandou o arquivo "como documento", então ele veio inteiro, com EXIF.

Rafa: *"Me diz o que o arquivo **afirma** sobre si mesmo. Depois a gente vê o que dá para acreditar."*

Neste notebook você vai:

1. **Carregar a imagem com custódia** (hash antes de tudo).
2. **Escrever um leitor de EXIF só com `struct`**, sem exiftool e sem Pillow.
3. **Checar hora local × GPS UTC** e **sinal × `Ref`** das coordenadas.
4. **Extrair a miniatura esquecida** e ver o que a edição escondeu.
5. **Calcular o azimute do sol** para checar sombras, offline.
6. **Jogar o quiz "confio ou não confio?"** e gerar sua linha do `points.csv`.

🐍 Só biblioteca padrão. Tudo é fictício (domínios `.example`, coordenadas de exercício).

## 1. Carregar a evidência, com hash primeiro

A imagem fica no repositório em `caso-continuo/dados/dia-27/`. O notebook tenta baixar do GitHub; se não houver internet, usa a **cópia embutida** abaixo (plano B). Nos dois casos, o primeiro gesto é o mesmo do Passo 1 do original: calcular o SHA-256 antes de olhar.

In [ ]:
import base64                      # para decodificar a cópia embutida (plano B)
import hashlib                     # SHA-256: a impressão digital do arquivo
import urllib.request              # para tentar baixar do GitHub (plano A)

URL = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/"
       "main/caso-continuo/dados/dia-27/aviso-agencia-centro.jpg")

# Plano B: o mesmo arquivo, em base64, dentro do notebook. Funciona sem internet.
EMBUTIDO = (
    "/9j/4Q3URXhpZgAASUkqAAgAAAAHAA8BAgATAAAAHAEAABABAgALAAAALwEAABIBAwABAAAAAQAAADEBAgAVAAAAOgEAADIBAgAU"
    "AAAATwEAAGmHBAABAAAAYgAAACWIBAABAAAAjAAAAPIAAAADAAOQAgAUAAAAYwEAABGQAgAHAAAAdwEAADGkAgANAAAAfgEAAAAA"
    "AAAIAAAAAQAEAAAAAgMAAAEAAgACAAAATgAAAAIABQADAAAAiwEAAAMAAgACAAAAVwAAAAQABQADAAAAowEAAAcABQADAAAAuwEA"
    "AB0AAgALAAAA0wEAAB8ABQABAAAA3gEAAAAAAAADAAMBAwABAAAABgAAAAECBAABAAAA5gEAAAICBAABAAAA5gsAAAAAAABGYWJy"
    "aWNhbnRlIEV4ZW1wbG8AQ2VsdWxhciBYMQBFZGl0YUZvdG8gT25saW5lIDIuMQAyMDI2OjEwOjA0IDIyOjQxOjA1ADIwMjY6MTA6"
    "MDQgMTY6MDM6MTEALTAzOjAwAFNOLUZJQ1QtMDA0MgAUAAAAAQAAAAkAAAABAAAAAAAAAAEAAAAsAAAAAQAAADYAAAABAAAAAAAA"
    "AAEAAAATAAAAAQAAAAMAAAABAAAACgAAAAEAAAAyMDI2OjEwOjA0ADQAAAAKAAAA/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAoH"
    "BwgHBgoICAgLCgoLDhgQDg0NDh0VFhEYIx8lJCIfIiEmKzcvJik0KSEiMEExNDk7Pj4+JS5ESUM8SDc9Pjv/2wBDAQoLCw4NDhwQ"
    "EBw7KCIoOzs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozv/wAARCAB4AKADASIAAhEBAxEB"
    "/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKB"
    "kaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJ"
    "ipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEB"
    "AQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMz"
    "UvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaX"
    "mJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDtaKKK"
    "9A4QooooAKKKKACiiigAooooAKKKKACiiigAooooAKKKKACijIzjPNFABRRRQAUVS1HWLDSvL+2z+V5udnyk5x16D3ql/wAJfoP/"
    "AD/f+Qn/AMKpQk9UhXSNqisX/hL9B/5/v/IT/wCFH/CX6D/z/f8AkJ/8Kfs59g5l3NqisX/hL9B/5/v/ACE/+FH/AAl+g/8AP9/5"
    "Cf8Awo9nPsHMu5tUVi/8JfoP/P8Af+Qn/wAKP+Ev0H/n+/8AIT/4Uezn2DmXc2qKxf8AhL9B/wCf7/yE/wDhR/wl+g/8/wB/5Cf/"
    "AAo9nPsHMu5tUVi/8JfoP/P9/wCQn/wo/wCEv0H/AJ/v/IT/AOFHs59g5l3NqisX/hL9B/5/v/IT/wCFH/CX6D/z/f8AkJ/8KPZz"
    "7BzLubVFYv8Awl+g/wDP9/5Cf/Cren63p2qyPHZXHmtGNzDYRgfiKThJatBdGL4nuJNJu0u4GJkuAw64CgKq9uT6/lW9pkMsVkpm"
    "nMzSHzMnsCBx1rnPHnSy/wC2n/stdTa/8ecH/XJf5CueLbqSQlJuTRLRRRWxRxfxD/5h/wD20/8AZa4uvRvFHh+/8QTWkOnrGzQq"
    "7NvfbwSorD/4Vt4j/wCeVt/3/FdlKtTjBKUlcylTm3dIwrPThdQiQu4yxHypuAxjqc9Tngd6tjQkxl7l0XyvNJaMAAYPykk/eGOR"
    "Wn/wrfxJ/wA87b/v+KP+Fb+JD/yztv8Av+Kp16f86Eqcv5TN/sBSWC3XQIVYqAGBUtkc5wAKj/sZGyiXJaTcyr8g2vyoXBz33Ctb"
    "/hW/iT/nnbf+BAo/4Vt4k/55W3/f8Uvb0/50P2cv5TI/sUedHGtzuDqxyq5IIxgHnjIIPNTjw+geRXuivlluqryAAc/e9/8AGtD/"
    "AIVt4l/55W3/AH/FH/Ct/Ep/5Z25/wC24o9vT/nQezl/KZL6HsuEi+1Jhg25jgbSPug898r+dCaKXdUMkkbYXeHjxsLAn19gPqa1"
    "v+FbeJP+edt/3/FH/CuPEp/gt/8AwIFHt6f86F7OX8py0kbRSNG6lWU4IYYNNrqz8N/EhOTHbH/tuKT/AIVt4j/55W3/AH/FX9Yp"
    "fzIXsp9jlaK6r/hW3iP/AJ5W3/f8Uf8ACtvEf/PK2/7/AIo+sUv5kHsp9jla634e/wDH/e/9cV/9Cpn/AArbxH/zytv+/wCK2fDP"
    "hvUvD1/N/aCxL58PyeXJu6MM/wAxWdStTlFpSQ405p3aK3jzpZf9tP8A2Wuptf8Ajzg/65L/ACFct486WX/bT/2Wuptf+POD/rkv"
    "8hXmQ/iy+RUfjZLWJYahdTeLdUsZJd1vbxRtGm0fKSBnnrW3XN6X/wAj5rX/AFxi/kK1lujZdTq9M/5Ch/64N/6Eta7usalnYKo6"
    "knArI0z/AJCh/wCuDf8AoS1d1b/kGy/h/MVyVFerY6IO0Lk32u2/5+Iv++xR9rtv+fiL/vsVytFafV13I9s+x1El3AY2CXMQYggE"
    "sODVGxs3jkZY75Jl2MoYNlwDjb7cY/WsWtXQf9fN/uD+dKVLki2mCnzSSJn0qWTALoilApCE5X5skgn1/Ckk0eaWIb7hRNtw0gHU"
    "5c5/8eH5VLr+rf2Jo0+o+T53k7fk3bc5YDr+Ncb/AMLW/wCoN/5Mf/Y1zuq0dCpX2O5sbWS1RldwchR8pPJA5bnue9Wq88/4Wt/1"
    "Bv8AyY/+xrrvDet/8JBpC6h9n+z7pGTZv3dPfAqObmY3BxRpSTRRY8yRUz03HGaZ9rtv+fiL/vsVm6//AMsP+Bf0rHrphRUo3uYS"
    "qOLsdV9rtv8An4i/77FH2u2/5+Iv++xXK0Vf1ddyfbPsdckiSLuRwy+qnIrK1b/j/t/+uL/+hLVjRf8AkHL/AL7VX1b/AI/7f/ri"
    "/wD6EtZQVqti5O8LnEePOll/20/9lrqbX/jzg/65L/IVy3jzpZf9tP8A2Wuptf8Ajzg/65L/ACFbw/iy+Rwx+Nktc3pf/I+a1/1x"
    "i/kK6Sub0v8A5HzWv+uMX8hWkt0bR2Z1emf8hQ/9cG/9CWrurf8AINl/D+Yqlpn/ACFD/wBcG/8AQlrUuYBdW7QsxUNjkVy1Harc"
    "6IK9OxylFbf9gxf893/IUf2DF/z3f8hW/todzH2cjErV0H/Xzf7g/nU39gxf893/ACFWbLTksndlkZ9wxyKipVjKLSLhTkpXZk+P"
    "f+RNvv8AgH/oa143Xsnj3/kTb7/gH/oa143Xnz3PQpfCFeufDf8A5FGP/rvJ/OvI69c+G/8AyKMf/XeT+dENwq/CaWv/APLD/gX9"
    "Kx66W9sEvtm6Rk2Z6Drmqv8AYMX/AD3f8hXoU6sYxSZ584ScroxKK2/7Bi/57v8AkKP7Bi/57v8AkKv20O5Hs5E2i/8AIOX/AHzV"
    "fVv+P+3/AOuL/wDoS1oWlstpAIVYsAScms/Vv+P+3/64v/6EtYQd6t0bSVqdjkfGGnXl+LT7JbvNs37tvbOMV0NupS1hVhgrGoI9"
    "DipKK6lBKTl3ORRs2wrm9L/5HzWv+uMX8hW/9oi/57J/30K5SLU7fTfGerXFx5hjljjVWjTdk4FYPERdrJnSqMkdvpn/ACFD/wBc"
    "G/8AQlrZriovEOnNJ5sWovA4XaTsIyDz3X2q7Hq4lz5etb8dcPH/AIVz1JqUrm0INKx1FFc39um/6Cj/APfUf+FH26b/AKCkn/fU"
    "f+FRzIqx0lFc39um/wCgpJ/31H/hR9um/wCgpJ/31H/hRzILDvHv/Im33/AP/Q1rxuvWr8JqdnJZ3moPLBJjcm9BnByOQPUVjf8A"
    "CIaB/ef/AL/1nLVmsGoqzPPq9c+G/wDyKMf/AF3k/nWJ/wAIhoH95/8Av/WvpsUWkWYtLG+eKEMWC70PJ68kUR0Y5yUlZHXUVzf2"
    "6b/oKSf99R/4Ufbpv+gpJ/31H/hWnMjGx0lFc39um/6Ckn/fUf8AhR9um/6Ckn/fUf8AhRzILHSVkat/x/2//XF//Qlql9um/wCg"
    "pJ/31H/hTDOHlEkt75rKpUb2XgHBPQD0FVCajJNkyg2rE9FRfaIv+eyf99Cj7RF/z2T/AL6FdX1mHZmHsJHEYpaKKwNwpMA9hS0U"
    "AIFUdFA+goxS0UAJijFLRQAmKMUtFACYoxS0UAJijFLRQAmKMUtFACYoxS0UAJijFLRQAUUUUAFFFFABRRRQAUUUUAFFFFABRRRQ"
    "AUUUUAFFFFABRRRQAUUUUAf/2f/bAEMACgcHCAcGCggICAsKCgsOGBAODQ0OHRUWERgjHyUkIh8iISYrNy8mKTQpISIwQTE0OTs+"
    "Pj4lLkRJQzxINz0+O//bAEMBCgsLDg0OHBAQHDsoIig7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7Ozs7"
    "Ozs7Ozs7O//AABEIAIwA8AMBIgACEQEDEQH/xAAfAAABBQEBAQEBAQAAAAAAAAAAAQIDBAUGBwgJCgv/xAC1EAACAQMDAgQDBQUE"
    "BAAAAX0BAgMABBEFEiExQQYTUWEHInEUMoGRoQgjQrHBFVLR8CQzYnKCCQoWFxgZGiUmJygpKjQ1Njc4OTpDREVGR0hJSlNUVVZX"
    "WFlaY2RlZmdoaWpzdHV2d3h5eoOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4eLj"
    "5OXm5+jp6vHy8/T19vf4+fr/xAAfAQADAQEBAQEBAQEBAAAAAAAAAQIDBAUGBwgJCgv/xAC1EQACAQIEBAMEBwUEBAABAncAAQID"
    "EQQFITEGEkFRB2FxEyIygQgUQpGhscEJIzNS8BVictEKFiQ04SXxFxgZGiYnKCkqNTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZn"
    "aGlqc3R1dnd4eXqCg4SFhoeIiYqSk5SVlpeYmZqio6Slpqeoqaqys7S1tre4ubrCw8TFxsfIycrS09TV1tfY2dri4+Tl5ufo6ery"
    "8/T19vf4+fr/2gAMAwEAAhEDEQA/AO1orzH/AITDXv8An/8A/IKf/E0f8Jhr3/P/AP8AkFP/AImvY+rTPN9qj06ivMf+Ew17/n//"
    "APIKf/E0f8Jhr3/P/wD+QU/+Jo+rTD2qPTqK8x/4TDXv+f8A/wDIKf8AxNH/AAmGvf8AP/8A+QU/+Jo+rTD2qPTqK8x/4TDXv+f/"
    "AP8AIKf/ABNH/CYa9/z/AP8A5BT/AOJo+rTD2qPTqK8x/wCEw17/AJ//APyCn/xNH/CYa9/z/wD/AJBT/wCJo+rTD2qPTqK8x/4T"
    "DXv+f/8A8gp/8TR/wmGvf8//AP5BT/4mj6tMPao9OorzH/hMNe/5/wD/AMgp/wDE0f8ACYa9/wA//wD5BT/4mj6tMPao9OorzH/h"
    "MNe/5/8A/wAgp/8AE0f8Jhr3/P8A/wDkFP8A4mj6tMPao9OorzH/AITDXv8An/8A/IKf/E0f8Jhr3/P/AP8AkFP/AImj6tMPao9O"
    "orzH/hMNe/5//wDyCn/xNH/CYa9/z/8A/kFP/iaPq0w9qj06ivMf+Ew17/n/AP8AyCn/AMTR/wAJhr3/AD//APkFP/iaPq0w9qj0"
    "6ivMf+Ew17/n/wD/ACCn/wATR/wmGvf8/wD/AOQU/wDiaPq0w9qj06ivMf8AhMNe/wCf/wD8gp/8TR/wmGvf8/8A/wCQU/8AiaPq"
    "0w9qj06ivMf+Ew17/n//APIKf/E0f8Jhr3/P/wD+QU/+Jo+rTD2qPTqK8x/4TDXv+f8A/wDIKf8AxNH/AAmGvf8AP/8A+QU/+Jo+"
    "rTD2qMWiiiu45wp8MMtzOkMKF5JCFVR1JplXdIuYrTUVknJWNkkiZlGSgdGTcB7bs/hSei0GtxTo18Jo4hEjmQMVdJkZML97Lg7R"
    "jvk8Uf2Lf+Zs8qPGzzPM85PL25xnfnb1469eKt20ljaQy2R1HzBdQujTIr+VEdyMpwQG52YYgdCOuKn+12H/AAj/APYX25M+d9p+"
    "1bH8vdjbs+7uxjnOOv51m5S/pF8qM9dD1BjKDHEgiKBmknjRSXBK4JYBshSRjPSmTaPe28AmnSKIEMQsk6K52sVPyE7vvKR07Vqy"
    "6vYQ6ObJUivWhMCp5quFk2+ezMMEHAMgUZ6jt6R6jf22paXbyTTWj3qxSeaZFmEu8zSP8u0bOQw6+po5p32DliULnQ9RtJRFLChf"
    "zvIKxzJJtk/unaTg8Hr6H0oi0S/luprVUhE0G7zEe4jXG0EkjLDIABORkYrduNd0x7+/uoliTbfPcxKqyf6UcP5bNk/KVJzxt+8e"
    "PTPGpWcmox3pKQl9OmhkjQMQj+S8aAZyTkbOcnqc0lObWw3GPczptKvLeEzPGpjCB9ySK427ymeCeNykf/rFH9lX3lTSeR8sEaSy"
    "HcPlVwCp69wc464+laVtfWTiwtbi48uBrB7a5fYT5ZM0jqcY5wSh4zU8Ou2XnnzI4zHe3Ennlg+YIivlqBggEqrN2PUd6blPsLlj"
    "3MW10y8vIjJBEGXJUZdVLkDJCgnLHBHAz1HrUo0O/MlvGEhMlyYxFH9oj3t5gBT5d2RkMOvrVzR7uzhtVhvLmJoBMWkgkiYtjA+a"
    "JwMqxx3IHC5z2urqGmDVdJvvOswLc2fmsFn84eWqB8jGzjaenpQ5yT2BRjYxxoWoMCypAUCFzILmLYAGVT827GcuvGc8io20m+UX"
    "JMHFoqtKQykAN0I5+bI54zxz0rWg1mwXR5YjbwIzQOgsx5uxnMsLbs7s5Kof4v8AlmPXmdtb05LhEURyQ3bAXDFZAbaMxCMKoyM7"
    "AXAyD2780uefYfLHuc7BZXNzbz3EMe6O2UNK2QNoJwPr+HoT2qT+y7o2L3o8loYwGcrcRllBOBlQ27qfStHStSsrG2htZVjkW5lc"
    "XUhD5iQrsBABAJAZz0PUd6gu/siaPDb2uoQPgCWaMJIHeQ8Y5XGFBx1/vHvVc0r2sTZWK66NqDXUlsLfMsd0toy714lYsAuc+qtz"
    "0461Srql1fTotZN4LsMl1rEN84EbZgRWckNxyf3n8OfumucubeKAqI7yG5z1MQcbfruUUQk29UEklsQUUUVoQFFFFABRRRQAUUUU"
    "AFFFFABRXtGg6Fo83h7TZZdJspJHtImZ2t0JYlBkk45NX/8AhHdD/wCgNYf+Aqf4V5v9ox/lOz6rLueEUV7v/wAI7of/AEBrD/wF"
    "T/Cj/hHdD/6A1h/4Cp/hR/aMf5Q+qy7nhFFe7/8ACO6H/wBAaw/8BU/wo/4R3Q/+gNYf+Aqf4Uf2jH+UPqsu54RRXu//AAjuh/8A"
    "QGsP/AVP8KP+Ed0P/oDWH/gKn+FH9ox/lD6rLueEUV7v/wAI7of/AEBrD/wFT/CoG0zwutv9oax0kQ7ivmGKLbkHBGcYzkEUf2hF"
    "/ZYfVX3PD6K91TQdAljWSPSdOdGGVZbaMgj2OKhTTfC0iSullpDLCCZSsUREYHXdxx0PWn/aEf5WL6q+54hRXtqWHhSQKUtNHcM4"
    "jUrHEcseij3ODxTp9N8LWwY3FlpEQRtrGSKJcHAODkdcEH6Gj6+v5WH1Z9zxCivaxaeEWZUFvopZ8bQEiy2Tjj8QRUq6X4Ye5Nsl"
    "hpLTr1iEMZYfhjNH19fysPqz7nh9Fe7/APCO6H/0BrD/AMBU/wAKP+Ed0P8A6A1h/wCAqf4Uv7Rj/KP6rLueEUV7v/wjuh/9Aaw/"
    "8BU/wo/4R3Q/+gNYf+Aqf4Uf2jH+UPqsu54RRXu//CO6H/0BrD/wFT/Cj/hHdD/6A1h/4Cp/hR/aMf5Q+qy7nhFFe7/8I7of/QGs"
    "P/AVP8KP+Ed0P/oDWH/gKn+FH9ox/lD6rLueEUV7v/wjuh/9Aaw/8BU/wo/4R3Q/+gNYf+Aqf4Uf2jH+UPqsu54RRXu//CO6H/0B"
    "rD/wFT/Cj/hHdD/6A1h/4Cp/hR/aMf5Q+qy7nhFFe7/8I7of/QGsP/AVP8KP+Ed0P/oDWH/gKn+FH9ox/lD6rLuJ4d/5FnS/+vOH"
    "/wBAFaVZvh3/AJFnS/8Arzh/9AFaVeOd4UVT1DUPsPl/ut+/P8WMYx7e9U/7f/6df/In/wBatFTlJXSIc4p2ZsUVj/2//wBOv/kT"
    "/wCtR/b/AP06/wDkT/61P2M+wvaR7mxRWP8A2/8A9Ov/AJE/+tR/b/8A06/+RP8A61HsZ9g9pHuaF9JLDYzPAheYIfLUDOW7frWB"
    "bWd1pGpRpKjTW0UZKSQ2zlUdlVQdoJJx5bZwf+WnbNX/AO3/APp1/wDIn/1qP7f/AOnX/wAif/WrSMKkVaxLlB63G2a3Nx4dubZI"
    "DaXLJOEXy2jAJdwpGScZxnGTjI7YqG+2Xmj3cdtps6SJp0sS7omUqSoxGBj5unbI4963YZPNgjkxjeobHpkU+s+ezvYvlujn7yGX"
    "V1s4Zi7gXRLOtpJD5Y8mQBvnz0bHPrio0S9uNB8QCe1kW5m8xQgQ/vGFuiZUdwWU4rpKKPa26C5Dn9Xtbya71IxLmB7GJJF8os0o"
    "DS7lQg8Ng+h6ii3jkNhYaf8AZplu7eaNpJDEwUFWy778YO4buhz8/wBa6Cij2jtYfJrcKKKKyLCiiigAorH/ALf/AOnX/wAif/Wo"
    "/t//AKdf/In/ANatfYz7GftI9zYorH/t/wD6df8AyJ/9aj+3/wDp1/8AIn/1qPYz7B7SPc2KKx/7f/6df/In/wBaj+3/APp1/wDI"
    "n/1qPYz7B7SPc2KKx/7f/wCnX/yJ/wDWqzY6n9tnMfk7MLuzuz3Ht70nSmldoanFuyL9FFFZlmb4d/5FnS/+vOH/ANAFaVZvh3/k"
    "WdL/AOvOH/0AVpUAY+v/APLv/wAC/pWPWxr/APy7/wDAv6Vj130fgRyVPiYUUUVqZhRRRQAUUUUAdVZ/8eUH/XNf5VNUNn/x5Qf9"
    "c1/lU1eZLdnctjO1bX9M0Pyv7RufI87Pl/u2bOMZ+6D6is7/AIT3wx/0E/8AyBJ/8TXPfFb/AJhX/bb/ANkrzysnJpm8aaaueyf8"
    "J74Y/wCgn/5Ak/8AiaP+E98Mf9BP/wAgSf8AxNeN0UudleyR7J/wnvhj/oJ/+QJP/iau6X4m0fWrlrbT7zzpVQuV8t1wuQM8gdyK"
    "8Orsvhf/AMjJcf8AXm3/AKGlNSbZMqaSueqUUUVoYnH0UUV6hwBRRRQAUUUUAFaWhf8AH6//AFzP8xWbWloX/H6//XM/zFZ1fgZc"
    "PiRvUUUV552Gb4d/5FnS/wDrzh/9AFaVYGg63pEPh7TYpdUso5EtIlZGuEBUhBkEZ4NX/wDhINF/6DFh/wCBKf40uZdzT2NT+V/c"
    "V9f/AOXf/gX9Kx6t63rWlS+R5ep2b43Z2zqcdPesr+1NP/5/7b/v8v8AjXdSnHkWpy1KNXmfuv7i1RVX+1NP/wCf+2/7/L/jR/am"
    "n/8AP/bf9/l/xrXnj3I9hV/lf3Fqiqv9qaf/AM/9t/3+X/Gj+1NP/wCf+2/7/L/jRzx7h7Cr/K/uLVFVf7U0/wD5/wC2/wC/y/40"
    "f2pp/wDz/wBt/wB/l/xo549w9hV/lf3HaWf/AB5Qf9c1/lU1ZVrr2jLaQq2rWIIjUEG5Tjj61L/wkGi/9Biw/wDAlP8AGvNclfc7"
    "FRqW+F/ccZ8Vv+YV/wBtv/ZK88ru/iZf2V//AGZ9ju4LnZ5u7yZA+3OzGcdOhrhcH0NYSaudVOlU5fhf3CUUuD6GjB9DU3Rfsp/y"
    "sSuy+F//ACMlx/15t/6Glcdg+hrrPhzd21j4gnlu7iK3jNqyhpXCAnenGT34NVFq5M6VTlfuv7j1mis//hINF/6DFh/4Ep/jR/wk"
    "Gi/9Biw/8CU/xrbmXc5fY1P5X9xz9FVf7U0//n/tv+/y/wCNH9qaf/z/ANt/3+X/ABr0+ePc4vYVf5X9xaoqr/amn/8AP/bf9/l/"
    "xo/tTT/+f+2/7/L/AI0c8e4ewq/yv7i1RVX+1NP/AOf+2/7/AC/40f2pp/8Az/23/f5f8aOePcPYVf5X9xarS0L/AI/X/wCuZ/mK"
    "w/7U0/8A5/7b/v8AL/jWho2saXFdu0mpWiDyyMtOo7j3rOpOPI9SoUKvMvdf3HVUVn/8JBov/QYsP/AlP8aP+Eg0X/oMWH/gSn+N"
    "cHMu51+xqfyv7jw+p4LK4uIJ54o90duA0jZA2gnA+v4eh9Kgra0rUrOytobaVEdbmRhcuQ+YkK7ARggEgM56HqK81JN6n39SUoxv"
    "FXZWj0HUJkgeNbdxcSeXFi6iyz8fKBuzn5l+maYmi30rSrCkUxihaZ/KnjfagIBPDH1HHWrml39rbHRvNlx9l1Np5TtJ2ofJ56c/"
    "cbgc8U+G/t9ON1Nay2iStbqIhbCYguJo258z2U+3FXaJg6lZO1vwfe3fsZ0OlXk6B0jQI0PnhnlRBs3+XnLEY+bjHWlXRtQaYRLb"
    "5ZpY4lw64ZpM7MHOCDtPPT3rYur3SZ43t7a7SCOTTfLXzEfEbm580pwpJwvGenFOs9X0+3+wQG4ylnd2jGTY3zqjTM7AYzgGQAdy"
    "O1HLHuS61Wzaj+DMM6XeiO8cwELYMFuDuH7slto7888cVP8A8I/qRvxYLFC90WZfKS5jZgVBLA4bjAB61oT6tZPpN1Ekv7+8tlMo"
    "2nmUPDxnHojtnp82OtSXuoad/b13fwT2oSYXe1oVn3sXjcKW3DAOSOnr6Ucse4e1ra+736Pol+tzCj0+5lRJAqIkjuitJKqDcoUs"
    "PmIx95fzqxJoGoRSyRyJAhibY5a6iCq3I2k7sZ4PHXirGs6haX2mWbRP/pLzSzXMe0ja7LGuc9DuKFuOmcVLqV3ZapNexrepAv8A"
    "aE1xFJIj7ZEcj0BII2g8juaVol+0quztbe+j01/r8zKTTrp7mW28rZJDkyCRggTBxySQByQPxFPOkX4nhhMH7yefyIxuX5n+XjOe"
    "nzrz0561eXUYX1a4ljulhjMCQq08O9JQoRfnXBIzt3dDg4+tXI9R0pbuzljnWGOwvftOza5EvyxZCcHA3RnG7HBFCjETq1V06dn2"
    "/wAzDTTbuS1+0rGDHtLj51DFQcFguckDnkDHB9KsP4f1JJ2gMUW9WdXxcRkIVUswYhsLgKx5x0NT293af2QIbm4jlCQyKkTRMJY3"
    "OduxwMFckEgnu3Hc3TrtpJ4hvSFt4rGeS5cuVkxMWjkVC/JYff8A4QPvGmlEUqtZN2W1+j+RjHSL37PcXCxxyRWu0zPHMjhd2cdC"
    "c9D06d6RNIvZJCgiVSFjYl5VUYddyck4yQc469fStKDUrTT4Z1jNvIJLiFmhtxJseMJKsgy/PRx+fHSppr6w/tC4S3v4XtvLtowL"
    "m3YxzBIgpPA3KwIOCMdTz6nLEHVq3at+D8v83/SZk/2NqHllzAFIDHy2kUSELnJCE7iBg9B2PpVGujlvtMfW4dXhuikVts22sisZ"
    "G8sAKM424YKMknuevfnKmSS2NqU5y+JBRRRUmwUUUUAFFFFABRRRQAUUUUAFFFFABXQ6Nptrc6dayTWsEglupUmkeYrIsapGfkXc"
    "NxG5jjB7VDbeDPEF3bRXMGn74pkDo3nRjKkZBwW9Kuw+FPF1uIBFZbRbymaL97Edrnbk9efurweOK0jGSeqOGriaMo2jVivn/wAE"
    "jfS7Yz28X2O2WH7JHO7pckzORAJGG3ecZII+73FQRWNtdJBO1gkCy292cRTFkZo4S6n7xYMCRkH2960T4e8ZkxN9giDwhFSRfs4c"
    "BQAvzDk8ADrQ/h7xo7Qn7Eii3ZmjVDAiqWADcAgYIAyOnX1NVy+T+4wVeH/P2P8A4F5P/gGV/Z9r/ZHn+V+8/s3z924/f+1+Xnr/"
    "AHOMdO/WjSNKivNNupJVQyvlLctMFIZVLHCkjcT8i4wfvVqr4d8ZrM8n2CIl4hCUP2cpsDBsBD8o5APA61GfC/jAzwTCwRHt38yL"
    "y2gUK2Qc4Bx2FHK+zL+sQs17WOrv8X4Elto1hLc2kU1nAIXFl80c7GR2k8suHXcdoIZ+cDtjrWJaC2TSJ57mxhfrFDIWcO0h78Nj"
    "Cg56f3R3rai8OeNIbr7THabZQsahvMh6R7dnGccbF/L3NNHhjxeLJLM6bC0KKyqGFuzKCcnDHnv1zQ0+34ExrU1vVi9vtfeZmsWs"
    "VrbwiG0tEVreBzILgmUs0Ss2U3nHJP8ADV3TtM0+8lsY0t45ozLbrcESusqFmVX3KTjaSxAK+q5OaluPC/i66hWKbTYWCoqBv9HD"
    "7VACjcOeAAOtPPhzxp5Sxi027AoDrJCHwuNoLA7iBgYBPYego5Xfb8Adem4Je1jf/EZVhpsF1baczKAZbuZZmL7f3SJGx6nAwC/P"
    "vUk2m20Go3kphR7eG1FzHCku5TuKrt3A5IVnOSDk7Tz3rQl8N+Mpid1ioBRkwjQIMN97gEDJ7nrTIvC3jCFoWSyA8mMxoDJCRsJJ"
    "KkE4Iyx4OetLlfYr6xDf2sf/AALzv/wDPt7O21GK7eK1W3dbISIHl2Rh/ORCVZj02kjknnPtViDR7eO4sDPFDIgsnuLgfaVCSESy"
    "Ko3hsDOEHBH51X1q113SP+QpGIBeReUFXyypRWVsALwoB2njH86zW1G4a1W2Mv7pUEYXaPuhi+M4z95if/1Ci1ug/bQa0qr7/K29"
    "zZ/sa1S6VWXfHaXMguSGP7yILvQ5z/EquBj2qDWbWG1ghENnaRq9vA/mLcFpdzRqzZQucDJP8NZ/9r3flTRef8k8aRyDaPmVAAo6"
    "dgMZ60XGq3F1CsUxhYKioG+zoH2qAFG4DPAAHWhrTYI1YqSbqL7/AOv+GOig0rTGvrG2ntrdEmWz2lLgmWVn8suGXcdoIZznA6DF"
    "Zkdlat4fOpmAl48wGPccMxIxL16AHGOm7b6kVnDUrhb2G9E37+Dy/LfaPl2ABOMY4Cj8qu6VHq+sXH2TTh5zx2zIY/kUCInnrgdW"
    "z6557Ub9BKpBb1V0+195l0V0H/CCeJf+gb/5Hj/+Ko/4QTxL/wBA3/yPH/8AFVHJLsdP1vD/APPyP3o5+iug/wCEE8S/9A3/AMjx"
    "/wDxVH/CCeJf+gb/AOR4/wD4qjkl2D63h/8An5H70c/RXQf8IJ4l/wCgb/5Hj/8AiqP+EE8S/wDQN/8AI8f/AMVRyS7B9bw//PyP"
    "3o5+iug/4QTxL/0Df/I8f/xVH/CCeJf+gb/5Hj/+Ko5Jdg+t4f8A5+R+9HP0V0H/AAgniX/oG/8AkeP/AOKo/wCEE8S/9A3/AMjx"
    "/wDxVHJLsH1vD/8APyP3o5+iug/4QTxL/wBA3/yPH/8AFUf8IJ4l/wCgb/5Hj/8AiqOSXYPreH/5+R+9Ha20F9aW0VtBrN4kUKBE"
    "XZCcKBgDJj9Kkzqf/QcvP+/cH/xuuM/4TXUv+eFr/wB8N/8AFV29d7bR8JZEedT/AOg5ef8AfuD/AON0Z1P/AKDl5/37g/8AjdSU"
    "UuZjsiPOp/8AQcvP+/cH/wAbozqf/QcvP+/cH/xupKKOZhZEedT/AOg5ef8AfuD/AON0Z1P/AKDl5/37g/8AjdSUUczCyI86n/0H"
    "Lz/v3B/8bozqf/QcvP8Av3B/8bqSijmYWRHnU/8AoOXn/fuD/wCN0Z1P/oOXn/fuD/43UlFHMwsjL1XQhrflf2lqV5P5OfL4iXbn"
    "GfuoPQVnf8IJpf8Az8Xf/fa//E10tFS9RptbHNf8IJpf/Pxd/wDfa/8AxNH/AAgml/8APxd/99r/APE10tFFh8zOa/4QTS/+fi7/"
    "AO+1/wDiauaX4ah0a5a50+/vIJWQoW/dtlcg4wyHuBWzRQDbZHnU/wDoOXn/AH7g/wDjdGdT/wCg5ef9+4P/AI3UlFVzMmyI86n/"
    "ANBy8/79wf8AxujOp/8AQcvP+/cH/wAbqSijmYWRHnU/+g5ef9+4P/jdGdT/AOg5ef8AfuD/AON1JRRzMLIjzqf/AEHLz/v3B/8A"
    "G6M6n/0HLz/v3B/8bqSijmYWRHnU/wDoOXn/AH7g/wDjdGdT/wCg5ef9+4P/AI3UlFHMwsiPOp/9By8/79wf/G6M6n/0HLz/AL9w"
    "f/G6koo5mFkf/9k="
)

try:
    # timeout curto: se a rede estiver ruim, não queremos travar a aula
    with urllib.request.urlopen(URL, timeout=10) as resp:
        dados = resp.read()
    origem = "GitHub (raw)"
except Exception as erro:          # sem rede, 404, proxy... qualquer falha cai aqui
    dados = base64.b64decode(EMBUTIDO)
    origem = f"cópia embutida (rede indisponível: {type(erro).__name__})"

sha = hashlib.sha256(dados).hexdigest()   # hash ANTES de qualquer análise
print("Origem:", origem)
print("Tamanho:", len(dados), "bytes")
print("SHA-256:", sha)

# Guardamos a cópia de trabalho; o "original" é a variável dados, que não mexemos mais.
with open("aviso-agencia-centro.jpg", "wb") as f:
    f.write(dados)

## 2. Um leitor de EXIF com `struct`

Por dentro, um JPEG é uma sequência de **segmentos** que começam com `FF xx`. O EXIF mora no segmento **APP1** (`FF E1`), que começa com o texto `Exif\0\0` e, logo depois, contém um pequeno arquivo **TIFF**.

Um TIFF tem **diretórios de tags** (IFDs). Cada entrada tem 12 bytes: número da tag, tipo, quantidade e o valor (ou o *offset* onde o valor está, se ele não couber em 4 bytes).

| IFD | O que guarda |
|---|---|
| IFD0 | Make, Model, Software, data de modificação |
| ExifIFD | DateTimeOriginal, OffsetTimeOriginal, número de série |
| GPS IFD | latitude, longitude, Ref, hora UTC |
| IFD1 | a **miniatura** (thumbnail) |

In [ ]:
import struct                      # converte bytes em números (o coração do leitor)

# Nomes das tags que nos interessam. Tag = número de 16 bits definido na especificação EXIF.
NOMES = {
    0x010F: "Make", 0x0110: "Model", 0x0131: "Software", 0x0132: "ModifyDate",
    0x9003: "DateTimeOriginal", 0x9011: "OffsetTimeOriginal", 0xA431: "SerialNumber",
    0x0001: "GPSLatitudeRef", 0x0002: "GPSLatitude", 0x0003: "GPSLongitudeRef",
    0x0004: "GPSLongitude", 0x0007: "GPSTimeStamp", 0x001D: "GPSDateStamp",
    0x001F: "GPSHPositioningError",
    0x0201: "ThumbnailOffset", 0x0202: "ThumbnailLength",
}
TAMANHO_DO_TIPO = {1: 1, 2: 1, 3: 2, 4: 4, 5: 8, 7: 1}   # BYTE, ASCII, SHORT, LONG, RATIONAL, UNDEFINED


def achar_tiff(jpeg: bytes) -> int:
    """Devolve a posição onde o TIFF do EXIF começa, ou -1 se não houver EXIF."""
    pos = jpeg.find(b"Exif\x00\x00")      # assinatura do APP1 Exif
    return -1 if pos < 0 else pos + 6       # o TIFF começa 6 bytes depois


def ler_ifd(tiff: bytes, offset: int, ordem: str):
    """Lê um diretório de tags e devolve (dicionário de tags, offset do próximo IFD)."""
    n = struct.unpack(ordem + "H", tiff[offset:offset + 2])[0]   # quantas entradas
    tags = {}
    for k in range(n):
        e = offset + 2 + 12 * k                                  # cada entrada tem 12 bytes
        tag, tipo, qtd = struct.unpack(ordem + "HHI", tiff[e:e + 8])
        tam = TAMANHO_DO_TIPO.get(tipo, 1) * qtd                 # tamanho total do valor
        # Se o valor cabe em 4 bytes, ele está na própria entrada; senão, ali há um offset.
        if tam <= 4:
            bruto = tiff[e + 8:e + 8 + tam]
        else:
            ptr = struct.unpack(ordem + "I", tiff[e + 8:e + 12])[0]
            bruto = tiff[ptr:ptr + tam]
        # Traduz o valor bruto conforme o tipo
        if tipo == 2:                                            # ASCII: texto terminado em \0
            valor = bruto.rstrip(b"\x00").decode("ascii", "replace")
        elif tipo == 3:                                          # SHORT: inteiros de 16 bits
            valor = struct.unpack(ordem + "H" * qtd, bruto)
        elif tipo == 4:                                          # LONG: inteiros de 32 bits
            valor = struct.unpack(ordem + "I" * qtd, bruto)
        elif tipo == 5:                                          # RATIONAL: pares numerador/denominador
            nums = struct.unpack(ordem + "I" * (2 * qtd), bruto)
            valor = tuple(nums[i] / nums[i + 1] for i in range(0, len(nums), 2))
        else:
            valor = bruto
        tags[tag] = valor
    proximo = struct.unpack(ordem + "I", tiff[offset + 2 + 12 * n: offset + 6 + 12 * n])[0]
    return tags, proximo


def ler_exif(jpeg: bytes) -> dict:
    """Lê IFD0, ExifIFD, GPS e IFD1 e devolve tudo num dicionário com nomes legíveis."""
    i = achar_tiff(jpeg)
    if i < 0:
        return {}                                   # sem EXIF: provavelmente removido no upload
    tiff = jpeg[i:]
    ordem = "<" if tiff[:2] == b"II" else ">"       # II = Intel (little-endian), MM = Motorola
    ifd0_off = struct.unpack(ordem + "I", tiff[4:8])[0]
    ifd0, ifd1_off = ler_ifd(tiff, ifd0_off, ordem)
    todas = dict(ifd0)
    if 0x8769 in ifd0:                              # ponteiro para o ExifIFD
        todas.update(ler_ifd(tiff, ifd0[0x8769][0], ordem)[0])
    if 0x8825 in ifd0:                              # ponteiro para o GPS IFD
        todas.update(ler_ifd(tiff, ifd0[0x8825][0], ordem)[0])
    miniatura = b""
    if ifd1_off:                                    # IFD1 existe: há miniatura
        ifd1, _ = ler_ifd(tiff, ifd1_off, ordem)
        ini, tam = ifd1[0x0201][0], ifd1[0x0202][0]
        miniatura = tiff[ini:ini + tam]
    legivel = {NOMES[t]: v for t, v in todas.items() if t in NOMES}
    legivel["_miniatura"] = miniatura
    return legivel


exif = ler_exif(dados)
for nome, valor in exif.items():
    if nome != "_miniatura":
        print(f"{nome:22s}: {valor}")
print(f"{'(miniatura)':22s}: {len(exif['_miniatura'])} bytes")

> 💡 Compare com a saída do original (`exiftool -a -u -G1 -s`). É a mesma informação; aqui você viu **onde** ela mora. Se quiser conferir com a ferramenta de referência, a célula abaixo é opcional (o teste pula linhas com `!`).

In [ ]:
# OPCIONAL (Colab/Linux): instala e roda o exiftool de verdade para comparar.
!apt-get -qq install -y libimage-exiftool-perl > /dev/null
!exiftool -a -u -G1 -s aviso-agencia-centro.jpg
print("(Se você está fora do Colab, as linhas com ! foram puladas. Tudo bem.)")

## 3. Duas checagens que o original exige

**Hora:** `DateTimeOriginal` é o relógio local da câmera. Somando o `OffsetTimeOriginal`, chegamos ao UTC; o GPS dá o UTC do satélite. Se baterem, a hora sobe de confiança.

**Sinal:** graus/minutos/segundos não têm sinal. O sinal vem do `Ref`: `S` e `W` são negativos. Se o `Ref` estiver errado, o ponto pula de hemisfério.

In [ ]:
from datetime import datetime, timedelta, timezone

# ---- Checagem 1: hora local + offset  ×  hora UTC do GPS ----
local = datetime.strptime(exif["DateTimeOriginal"], "%Y:%m:%d %H:%M:%S")
sinal = -1 if exif["OffsetTimeOriginal"].startswith("-") else 1
hh, mm = map(int, exif["OffsetTimeOriginal"][1:].split(":"))
fuso = timezone(sinal * timedelta(hours=hh, minutes=mm))      # ex.: -03:00 = horário de Brasília
utc_pela_camera = local.replace(tzinfo=fuso).astimezone(timezone.utc)

h, m, s = exif["GPSTimeStamp"]                                # três racionais: hora, minuto, segundo
data_gps = datetime.strptime(exif["GPSDateStamp"], "%Y:%m:%d")
utc_pelo_gps = data_gps.replace(hour=int(h), minute=int(m), second=int(s), tzinfo=timezone.utc)

diferenca = abs((utc_pela_camera - utc_pelo_gps).total_seconds())
print("UTC pela câmera:", utc_pela_camera.isoformat())
print("UTC pelo GPS:   ", utc_pelo_gps.isoformat())
print("Diferença:", diferenca, "s ->", "BATE ✅" if diferenca <= 60 else "NÃO BATE ⚠️")

# ---- E a edição? ModifyDate vem do editor, não da câmera ----
print("\nSoftware:", exif.get("Software"), "| ModifyDate:", exif.get("ModifyDate"))
print("Foto tirada às", local.strftime("%H:%M"), "local; arquivo regravado às",
      exif.get("ModifyDate", "?")[-8:-3], "-> a faixa foi colada horas depois.")

In [ ]:
def dms_para_decimal(dms, ref):
    """(graus, minutos, segundos) + Ref ('N','S','E','W') -> decimal com sinal."""
    graus, minutos, segundos = dms
    valor = graus + minutos / 60 + segundos / 3600     # 1 grau = 60 minutos = 3600 segundos
    return -valor if ref in ("S", "W") else valor      # sul e oeste são negativos


lat = dms_para_decimal(exif["GPSLatitude"], exif["GPSLatitudeRef"])
lon = dms_para_decimal(exif["GPSLongitude"], exif["GPSLongitudeRef"])
print(f"Como o arquivo afirma:      lat={lat:.6f}  lon={lon:.6f}  (Ref: {exif['GPSLatitudeRef']}, {exif['GPSLongitudeRef']})")

# Minas Gerais inteira está no hemisfério SUL (latitude negativa, entre ~ -14 e ~ -23).
em_mg = -23 <= lat <= -14 and -51.5 <= lon <= -39.8
print("Cai dentro de Minas Gerais?", "sim" if em_mg else "NÃO ⚠️")

lat_invertida = dms_para_decimal(exif["GPSLatitude"], "S")
print(f"Se o Ref fosse 'S':         lat={lat_invertida:.6f}  lon={lon:.6f}")
print("\nLeitura: com 'N', o ponto cai no Oceano Atlântico, ao norte do Brasil;"
      " com 'S', cai no interior de MG.\nAs duas coisas não podem ser verdade ao mesmo tempo:"
      " o Ref foi adulterado (ou o editor errou). Confiança: CONTESTADA.")

## 4. A miniatura esquecida

O `Software` nomeia um editor online. O original manda, nesse caso, extrair a `ThumbnailImage` e comparar o enquadramento. Se a miniatura for diferente, ela é **anterior à edição**.

In [ ]:
from pathlib import Path

mini = exif["_miniatura"]
Path("aviso_thumb.jpg").write_bytes(mini)                 # = exiftool -b -ThumbnailImage


def dimensoes_jpeg(b: bytes):
    """Acha o segmento SOF0/SOF2 e lê altura e largura (sem biblioteca de imagem)."""
    i = 2
    while i < len(b):
        marcador, tam = b[i + 1], struct.unpack(">H", b[i + 2:i + 4])[0]
        if marcador in (0xC0, 0xC2):                         # Start Of Frame
            altura, largura = struct.unpack(">HH", b[i + 5:i + 9])
            return largura, altura
        i += 2 + tam                                         # pula para o próximo segmento
    return None

print("Imagem principal:", dimensoes_jpeg(dados), " proporção ~1,71 (recortada)")
print("Miniatura:       ", dimensoes_jpeg(mini), " proporção 1,33 (foto inteira de celular)")
print("SHA-256 da miniatura:", hashlib.sha256(mini).hexdigest()[:16], "...")

# No Colab, mostramos as duas lado a lado. Fora dele, os arquivos ficam na pasta.
try:
    from IPython.display import Image, display
    display(Image(data=dados)); display(Image(data=mini))
except ImportError:
    print("(Sem IPython: abra aviso-agencia-centro.jpg e aviso_thumb.jpg num visualizador.)")

> 🔍 **O que a miniatura mostra:** a fachada inteira, a calçada, um poste e **nenhuma faixa vermelha**. A foto é de verdade; a faixa foi acrescentada depois, no editor online. Isso é um achado sobre o **padrão** do golpe (reaproveitar foto real + colar o texto da isca), não sobre quem fotografou.

## 5. O sol, sem internet

No Passo 5 do original, o SunCalc confere se a sombra aponta para o lado certo. Abaixo, a aproximação de baixa precisão da posição do sol (a mesma família de fórmulas que a NOAA publica): erro de alguns décimos de grau, mais que suficiente para dizer "leste ou oeste".

In [ ]:
import math

def posicao_do_sol(lat, lon, quando_utc):
    """Devolve (azimute, elevação) em graus. Azimute: 0 = norte, 90 = leste, 180 = sul, 270 = oeste."""
    n = quando_utc.timestamp() / 86400 + 2440587.5 - 2451545.0      # dias desde 1/jan/2000 12h UTC
    L = (280.460 + 0.9856474 * n) % 360                             # longitude média do sol
    g = math.radians((357.528 + 0.9856003 * n) % 360)               # anomalia média
    lam = math.radians(L + 1.915 * math.sin(g) + 0.020 * math.sin(2 * g))  # longitude eclíptica
    eps = math.radians(23.439 - 0.0000004 * n)                      # inclinação do eixo da Terra
    ra = math.atan2(math.cos(eps) * math.sin(lam), math.cos(lam))   # ascensão reta
    dec = math.asin(math.sin(eps) * math.sin(lam))                  # declinação
    gmst = (18.697374558 + 24.06570982441908 * n) % 24              # tempo sideral de Greenwich (h)
    H = math.radians((gmst * 15 + lon) % 360) - ra                  # ângulo horário local
    la = math.radians(lat)
    elev = math.asin(math.sin(la) * math.sin(dec) + math.cos(la) * math.cos(dec) * math.cos(H))
    azim = math.atan2(-math.sin(H), math.tan(dec) * math.cos(la) - math.sin(la) * math.cos(H))
    return math.degrees(azim) % 360, math.degrees(elev)

# Usamos a coordenada CORRIGIDA (Ref = S), que é a hipótese coerente com a cena.
az, el = posicao_do_sol(lat_invertida, lon, utc_pelo_gps)
print(f"Sol às {utc_pelo_gps:%H:%M} UTC: azimute {az:.0f}°, elevação {el:.0f}°")
print("As sombras apontam para o lado oposto ao sol:", f"{(az + 180) % 360:.0f}°")
print("Na miniatura, a sombra do poste vai para a DIREITA da cena. Com o sol a oeste,"
      " a sombra vai para leste;\nse leste está à direita, quem fotografa olha para o norte e a fachada está virada para o sul.\nÉ uma hipótese a"
      " conferir em imagem de rua, não uma conclusão.")

## 6. 🎮 Quiz: confio ou não confio?

Para cada afirmação, responda **S** (dá para usar como achado, com a confiança indicada) ou **N** (não dá, ou não sozinho). `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
QUIZ = [
    ("A tag Software = 'EditaFoto Online 2.1' indica que o arquivo não é o original da câmera.", "S",
     "Software nomeando um editor é exatamente o sinal que o original manda observar."),
    ("A latitude 20°9'0\" N prova que a foto foi tirada no hemisfério norte.", "N",
     "O Ref contradiz a cena (fachada de MG). Contestado até conferir visualmente."),
    ("A hora local 16:03:11 com offset -03:00 bate com o GPS 19:03:10 UTC.", "S",
     "16:03:11 + 3h = 19:03:11 UTC; 1 s de diferença. Sobe a confiança na hora."),
    ("Como a hora bate, então as coordenadas também estão certas.", "N",
     "Um campo certo não valida outro. Cada campo tem seu modo de falhar."),
    ("A miniatura sem faixa mostra que a faixa foi acrescentada na edição.", "S",
     "Miniatura com enquadramento diferente é anterior à edição."),
    ("O SerialNumber permite descobrir quem é o dono do celular, então devo pesquisar isso.", "N",
     "É artefato para agrupar imagens. Atribuição de pessoa é trabalho da polícia."),
    ("Uma foto baixada do feed do Instagram sem EXIF prova que o EXIF foi apagado de propósito.", "N",
     "Grandes plataformas removem EXIF no upload. Ausência não é evidência de má-fé."),
    ("Uma coordenada isolada, sem nada visual que a confirme, é uma pista, não um achado.", "S",
     "Metadado é forte quando concorda com a cena e outras fontes; fraco sozinho."),
    ("Posso rodar exiftool -GPSLatitudeRef=S no arquivo de evidência para corrigir.", "N",
     "Nunca edite evidência. Registre a contradição nas notas; trabalhe em cópia."),
]

MODO_JOGO = True    # False = só mostra o gabarito

acertos = 0
for i, (afirmacao, certo, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {afirmacao}")
    if MODO_JOGO:
        r = ""
        while r not in ("S", "N"):                 # insiste até vir S ou N
            r = input("   Confio? (S/N): ").strip().upper()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end=" ")
    print(f"Gabarito: {certo}. {porque}")

if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)} — meta do Checkpoint BR: 7.")

## 7. 📝 Sua linha do `points.csv`

Preencha o formulário. O valor padrão já traz uma resposta razoável; **troque a confiança e a fonte com as suas palavras**.

In [ ]:
#@title Linha do points.csv { display-mode: "form" }
nome = "aviso-agencia-centro"  #@param {type:"string"}
metodo = "EXIF GPS (Ref contestado) + miniatura"  #@param {type:"string"}
confianca = "contestado (Ref N contradiz cena em MG)"  #@param ["alta", "média", "baixa", "contestado (Ref N contradiz cena em MG)"]
fonte = "arquivo recebido pela cooperativa em 2026-10-07; hash em hashes.txt"  #@param {type:"string"}
usar_lat_corrigida = True  #@param {type:"boolean"}

import csv
lat_final = lat_invertida if usar_lat_corrigida else lat
with open("points.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["name", "lat", "lon", "method", "confidence", "source", "sha256"])
    w.writerow([nome, f"{lat_final:.6f}", f"{lon:.6f}", metodo, confianca, fonte, sha])
print(open("points.csv", encoding="utf-8").read())
if usar_lat_corrigida:
    print("⚠️ Você usou a latitude corrigida: escreva nas notas que é HIPÓTESE sua, não o que o arquivo diz.")

try:
    from google.colab import files
    files.download("points.csv")
except ImportError:
    print("(Fora do Colab: points.csv ficou na pasta atual.)")

## ✅ Antes de fechar o Dia 27

- [ ] Hash calculado antes de qualquer análise
- [ ] Hora local × GPS UTC conferida (e anotada)
- [ ] Contradição do `GPSLatitudeRef` registrada como **contestada**
- [ ] Miniatura extraída e diferença de enquadramento descrita
- [ ] Quiz com pelo menos 7/9
- [ ] `points.csv` com método, confiança e fonte

**Amanhã (Dia 28):** o "comunicado oficial" em PDF que circula nos mesmos grupos. Ele tem três `%%EOF`, e a primeira versão ainda está lá dentro.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>